[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/18_feature_selection/01_filter_methods.ipynb)

# Bài 01 — Feature selection: vì sao và filter methods

**🎯 Mục tiêu**
- Vì sao chọn feature: ít overfit hơn, nhanh hơn, dễ diễn giải, rẻ hơn khi thu thập dữ liệu — và khi nào **không** cần.
- Ba họ phương pháp: **filter** (bài này), **wrapper** (bài 02), **embedded** (bài 03).
- Filter: variance threshold, tương quan (Pearson/Spearman), ANOVA F / chi², **mutual information** — cái nào bắt được quan hệ phi tuyến?
- Đa cộng tuyến giữa các feature: ma trận tương quan, **VIF**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/18_feature_selection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_regression, mutual_info_regression
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)
import warnings

warnings.filterwarnings("ignore", message="An input array is constant")

## 1. Dữ liệu có cấu trúc đã biết

Friedman #1: $y = 10\sin(\pi x_1 x_2) + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5 + \varepsilon$ — 5 feature có ích (có tương tác và phi tuyến), thêm 15 feature **nhiễu**, 3 feature **dư thừa** (bản sao nhiễu của $x_4$), và 2 feature **hằng số gần như không đổi**.

In [ ]:
X5, y = make_friedman1(n_samples=1500, n_features=20, noise=1.0, random_state=0)
redundant = X5[:, [3]] + rng.normal(0, 0.05, (1500, 3))
near_constant = np.c_[np.zeros(1500), (rng.random(1500) < 0.002).astype(float)]
X = np.hstack([X5, redundant, near_constant])
names = [f"x{i + 1}" for i in range(5)] + [f"noise{i}" for i in range(15)] + ["x4_copy1", "x4_copy2", "x4_copy3", "const0", "almost_const"]
df = pd.DataFrame(X, columns=names)
print(df.shape)

## 2. Vì sao chọn feature?

Với model tuyến tính và **ít dữ liệu**, feature nhiễu làm tăng variance (mỗi feature thêm một hệ số phải ước lượng):

In [ ]:
cv = KFold(5, shuffle=True, random_state=0)
true_cols = [names.index(c) for c in ["x1", "x2", "x3", "x4", "x5"]]
for n in [60, 200, 1500]:
    idx = rng.choice(1500, n, replace=False)
    all_r2 = cross_val_score(LinearRegression(), X[idx], y[idx], cv=cv, scoring="r2").mean()
    true_r2 = cross_val_score(LinearRegression(), X[idx][:, true_cols], y[idx], cv=cv, scoring="r2").mean()
    print(f"n = {n:4d}: R² dùng 25 feature = {all_r2:.3f} | chỉ 5 feature thật = {true_r2:.3f}")

Khi dữ liệu ít, bỏ feature nhiễu giúp rõ rệt; khi dữ liệu nhiều, model tự "lờ" chúng đi. Lợi ích khác của ít feature: model nhanh, dễ giải thích, **rẻ** (không phải thu thập/duy trì pipeline cho feature vô ích), ít rủi ro khi dữ liệu thay đổi.

> Với nhiều model hiện đại (regularization mạnh, gradient boosting), chọn feature thường cải thiện **ít** về độ chính xác — lý do chính là chi phí và diễn giải.

## 3. Filter methods: đánh giá từng feature độc lập với model

### 3.1 Variance threshold

In [ ]:
vt = VarianceThreshold(threshold=1e-3).fit(X)
print("bị loại (phương sai ≈ 0):", [n for n, keep in zip(names, vt.get_support()) if not keep])

### 3.2 Tương quan, F-test và mutual information

| Phương pháp | Đo gì | Bắt được |
|---|---|---|
| Pearson / F-test (`f_regression`, `f_classif`) | quan hệ **tuyến tính** | tuyến tính |
| Spearman | quan hệ **đơn điệu** | đơn điệu |
| Chi² (`chi2`) | phụ thuộc giữa feature không âm (đếm) và lớp | — |
| **Mutual information** | mọi dạng phụ thuộc $I(X; Y)$ | phi tuyến, không đơn điệu |

In [ ]:
F, _ = f_regression(X, y)
mi = mutual_info_regression(X, y, n_neighbors=10, random_state=0)      # n_neighbors lớn hơn → ước lượng MI ít nhiễu hơn
spearman = df.apply(lambda c: pd.Series(c).corr(pd.Series(y), method="spearman")).abs()
scores = pd.DataFrame({"F (tuyến tính)": F / F.max(), "|Spearman|": spearman / spearman.max(), "mutual info": mi / mi.max()}, index=names)
ax = scores.loc[names[:5] + ["noise0", "noise1", "noise8", "x4_copy1"]].plot.bar(figsize=(11, 4))
ax.set_ylabel("điểm (chuẩn hoá theo max)")
ax.set_title("x3 có quan hệ (x3 − 0.5)² — không đơn điệu: F và Spearman bỏ sót, MI bắt được")
plt.show()
scores.round(3).sort_values("mutual info", ascending=False).head(10)

- $x_3$ ảnh hưởng mạnh nhưng theo hình chữ U → tương quan tuyến tính/đơn điệu ≈ 0 → F-test và Spearman xếp nó **ngang feature nhiễu**; mutual information đặt nó rõ ràng **trên** mọi feature nhiễu. (MI là ước lượng có nhiễu — với `n_neighbors` mặc định = 3, đôi khi một feature nhiễu "may mắn" có MI ngang tín hiệu yếu.)
- Các bản sao `x4_copy*` đều được xếp hạng cao — filter đánh giá **từng feature riêng lẻ**, không biết chúng dư thừa nhau.
- Filter cũng không thấy **tương tác**: $x_1, x_2$ chỉ quan trọng qua $\sin(\pi x_1 x_2)$ — mỗi cái riêng lẻ có tín hiệu yếu hơn thực tế.

## 4. Đa cộng tuyến: VIF

**Variance Inflation Factor** của feature $j$: hồi quy $x_j$ theo các feature còn lại, $\text{VIF}_j = \frac{1}{1 - R_j^2}$. VIF > 5–10 → $x_j$ gần như là tổ hợp tuyến tính của feature khác → hệ số model tuyến tính không ổn định (chương 06).

In [ ]:
def vif(Xv):
    out = []
    for j in range(Xv.shape[1]):
        others = np.delete(Xv, j, axis=1)
        r2 = LinearRegression().fit(others, Xv[:, j]).score(others, Xv[:, j])
        out.append(1 / max(1 - r2, 1e-12))
    return np.array(out)


cols_vif = ["x1", "x2", "x3", "x4", "x5", "x4_copy1", "x4_copy2", "noise0"]
pd.Series(vif(df[cols_vif].to_numpy()), index=cols_vif).round(1).sort_values(ascending=False)

## 5. Filter trong Pipeline — chọn bao nhiêu feature?

In [ ]:
ks = [3, 5, 8, 12, 20, 25]
for name, score_func in [("F-test", f_regression), ("mutual info", lambda X_, y_: mutual_info_regression(X_, y_, n_neighbors=10, random_state=0))]:
    r2 = [cross_val_score(make_pipeline(StandardScaler(), SelectKBest(score_func, k=k), RidgeCV()), X[:300], y[:300], cv=cv, scoring="r2").mean()
          for k in ks]
    plt.plot(ks, r2, "o-", label=name)
plt.xlabel("k (số feature giữ lại)")
plt.ylabel("R² (CV, 300 mẫu)")
plt.legend()
plt.title("Chọn k bằng CV — SelectKBest nằm TRONG Pipeline")
plt.show()

> ⚠️ Nhắc lại chương 15: filter phải nằm **trong** Pipeline — chọn feature trên toàn bộ dữ liệu rồi mới CV là rò rỉ (ảo tưởng 87% trên dữ liệu nhiễu!).

## 🧠 Kiểm tra nhanh

1. Feature có quan hệ $y = x^2$ với $x$ đối xứng quanh 0: Pearson ≈ ? Mutual information ≈ ?
2. Vì sao filter methods không phát hiện được feature dư thừa?
3. VIF = 20 nghĩa là gì? Có ảnh hưởng tới Random Forest không?
4. Khi nào chọn feature giúp độ chính xác nhiều nhất?

## 📌 Tóm tắt

- Chọn feature vì: ít dữ liệu (giảm variance), chi phí, diễn giải.
- Filter: nhanh, độc lập model; F/Pearson chỉ thấy tuyến tính, MI thấy phi tuyến; không thấy dư thừa và tương tác.
- VIF phát hiện đa cộng tuyến; mọi bước chọn phải nằm trong Pipeline.

## 📚 Tài liệu tham khảo

- Guyon & Elisseeff (2003) — *An Introduction to Variable and Feature Selection*
- Kraskov, Stögbauer & Grassberger (2004) — *Estimating mutual information*
- scikit-learn User Guide — Feature selection